**Without and With Column Transformer**

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('covid_toy.csv')
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


**by the data we have differnt types of categories so we apply:**

*   fever & age     => Numerical
*   gender & city   => Nominal OneHotEncoding
*   cough           => Nominal Ordinal





In [16]:
#find null values and fill them

In [17]:
df.isnull().sum()

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [18]:
#here in the above data we see the fever column has 10 null values
#by using the simple imputer we fill the values

In [19]:
from sklearn.model_selection import train_test_split

In [20]:
x = df.iloc[:, 0:5]   # first 5 columns (features)
y = df.iloc[:, -1]    # last column (target)
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

**Difference by using with and without Column Transformer**
*   without using column transformer




In [21]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [22]:
#using Simple imputer to fill the missing values (fill the mean from the column)
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])
X_test_fever = si.fit_transform(X_test[['fever']])

In [29]:
# Ordinalencoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])
X_test_cough = oe.transform(X_test[['cough']])
X_train_cough.shape

(80, 1)

In [30]:
# OneHotEncoding -> gender,city
ohe = OneHotEncoder(drop='first',sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

X_train_gender_city.shape

(80, 4)

In [32]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape

(80, 1)

In [34]:
#put back columns into the dataframe
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)

X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_train_transformed.shape

(80, 7)

**WITH COLUMN TRANSFROMER**

In [35]:
from sklearn.compose import ColumnTransformer

In [38]:
transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
],remainder='passthrough')      # remainder is used to drop or overcome across remaining columns

In [39]:
transformer.fit_transform(X_train).shape

(80, 7)

In [40]:
transformer.fit_transform(X_test).shape

(20, 7)